In [15]:
"""
Refactored script for consumer account and transaction analysis,
feature engineering, visualization, and logistic regression model training.

Data files expected:
  - data/q2-ucsd-acctDF.pqt
  - data/q2-ucsd-consDF.pqt
  - data/q2-ucsd-trxnDF.pqt
  - data/q2-ucsd-cat-map.csv
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# -------------------------
# DATA LOADING & PREPROCESSING
# -------------------------

def load_data():
    """
    Load account, consumer, transaction, and category mapping data.
    
    Returns:
        tuple: (acct_df, cons_df, txn_df, cat_map_df)
    """
    acct_df = pd.read_parquet("data/q2-ucsd-acctDF.pqt")
    cons_df = pd.read_parquet("data/q2-ucsd-consDF.pqt")
    txn_df  = pd.read_parquet("data/q2-ucsd-trxnDF.pqt")
    cat_map_df = pd.read_csv("data/q2-ucsd-cat-map.csv")
    return acct_df, cons_df, txn_df, cat_map_df

def create_category_map(cat_map_df):
    """
    Create a dictionary mapping from numeric category code to category string.
    
    Args:
        cat_map_df (pd.DataFrame): DataFrame with a 'category' column.
        
    Returns:
        dict: Mapping of numeric code -> category.
    """
    return {i: cat for i, cat in enumerate(cat_map_df['category'])}

def preprocess_transactions(txn_df, cat_map):
    """
    Preprocess transaction data by mapping numeric categories to strings and adding a 'year_month' column.
    
    Args:
        txn_df (pd.DataFrame): Transaction data.
        cat_map (dict): Mapping for the 'category' column.
    
    Returns:
        pd.DataFrame: Processed transaction DataFrame.
    """
    txn_df = txn_df.copy()
    # Map numeric category values to category names.
    txn_df['category'] = txn_df['category'].apply(lambda x: cat_map[x])
    # Create a 'year_month' column (as Period for monthly aggregation)
    txn_df['year_month'] = pd.to_datetime(txn_df['posted_date']).dt.to_period('M')
    return txn_df

def filter_transactions_by_consumers(txn_df, consumer_ids):
    """
    Filter transaction data to only include rows for given consumer IDs.
    
    Args:
        txn_df (pd.DataFrame): Transaction data.
        consumer_ids (set): Set of consumer IDs to keep.
        
    Returns:
        pd.DataFrame: Filtered transaction DataFrame.
    """
    return txn_df[txn_df['prism_consumer_id'].isin(consumer_ids)].copy()

def get_train_df(cons_df):
    """
    Extract the training DataFrame containing consumers with non-null DQ_TARGET.
    
    Args:
        cons_df (pd.DataFrame): Consumer-level data.
    
    Returns:
        pd.DataFrame: DataFrame with columns 'prism_consumer_id' and 'DQ_TARGET'.
    """
    return cons_df[cons_df['DQ_TARGET'].notna()][['prism_consumer_id', 'DQ_TARGET']].copy()

# -------------------------
# FEATURE ENGINEERING
# -------------------------

def aggregate_transactions(txn_df):
    """
    Aggregate transactions by consumer and month to compute monthly income,
    spending, and net income.
    
    Returns:
        pd.DataFrame: DataFrame with columns:
            'prism_consumer_id', 'year_month', 'monthly_income',
            'monthly_spending', 'monthly_net_income'.
    """
    # Compute monthly income and spending separately by filtering then grouping.
    income = txn_df[txn_df['credit_or_debit'] == 'CREDIT']\
                .groupby(['prism_consumer_id', 'year_month'])['amount']\
                .sum().rename('monthly_income')
    spending = txn_df[txn_df['credit_or_debit'] == 'DEBIT']\
                .groupby(['prism_consumer_id', 'year_month'])['amount']\
                .sum().rename('monthly_spending')
    txn_agg = pd.concat([income, spending], axis=1).fillna(0).reset_index()
    txn_agg['monthly_net_income'] = txn_agg['monthly_income'] - txn_agg['monthly_spending']
    return txn_agg

def zscoring_normalize(df, column):
    """
    Apply Z-score normalization to a specified column and add a new column with suffix '_z_score'.
    
    Args:
        df (pd.DataFrame): DataFrame containing the column.
        column (str): Column name to normalize.
        
    Returns:
        pd.DataFrame: DataFrame with an additional column named '<column>_z_score'.
    """
    df = df.copy()
    mean_val = df[column].mean()
    std_val  = df[column].std()
    df[column + '_z_score'] = (df[column] - mean_val) / std_val
    return df

def compute_average_nmi(txn_agg):
    """
    Compute the average net monthly income (NMI) for each consumer and apply Z-score normalization.
    
    Args:
        txn_agg (pd.DataFrame): Aggregated transaction data.
        
    Returns:
        pd.DataFrame: DataFrame with 'prism_consumer_id', 'average_nmi', and 'average_nmi_z_score'.
    """
    avg_nmi = txn_agg.groupby('prism_consumer_id')['monthly_net_income']\
                     .mean().reset_index().rename(columns={'monthly_net_income': 'average_nmi'})
    avg_nmi = zscoring_normalize(avg_nmi, 'average_nmi')
    return avg_nmi

def compute_balance_features(acct_df, txn_agg):
    """
    Compute monthly balance features using account data and aggregated transaction data.
    
    Steps:
      1. Convert account balance dates to a monthly period and aggregate balances.
      2. Compute cumulative monthly net inflow for each consumer.
      3. Merge the aggregated transactions with account balance on consumer ID and month.
      4. For each consumer, determine a 'start_balance' using the earliest available balance (anchor)
         and the inflows prior to that anchor.
      5. Compute monthly balance as: monthly_balance = cumulative_inflow + start_balance.
    
    Returns:
        pd.DataFrame: DataFrame with columns 'prism_consumer_id', 'year_month', and 'monthly_balance'.
    """
    # Prepare account data by adding a monthly period column.
    acct_df = acct_df.copy()
    acct_df['year_month'] = pd.to_datetime(acct_df['balance_date']).dt.to_period('M')
    balance_df = acct_df.groupby(['prism_consumer_id', 'year_month'])\
                        .agg({'balance': 'sum'}).reset_index()
    
    # Compute cumulative net inflow for each consumer.
    txn_agg = txn_agg.sort_values('year_month')
    txn_agg['cumulative_inflow'] = txn_agg.groupby('prism_consumer_id')['monthly_net_income'].cumsum()
    
    # Merge transaction data with account balance on both consumer ID and month.
    merged = pd.merge(txn_agg, balance_df, on=['prism_consumer_id', 'year_month'], how='left')
    
    def find_real_start_balance(grp):
        """
        For a given consumer group, find the earliest available balance (anchor)
        and compute the consumer's starting balance.
        """
        anchor_rows = grp.dropna(subset=["balance"]).sort_values("year_month")
        if anchor_rows.empty:
            return pd.Series({"start_balance": 0.0})
        anchor_balance = anchor_rows["balance"].iloc[0]
        anchor_month   = anchor_rows["year_month"].iloc[0]
        inflow_before_anchor = grp.loc[grp["year_month"] <= anchor_month, "monthly_net_income"].sum()
        start_balance = anchor_balance - inflow_before_anchor
        return pd.Series({"start_balance": start_balance})
    
    # Use groupby.apply with group_keys=False to avoid the deprecation warning.
    real_starts = merged.groupby("prism_consumer_id", group_keys=False)\
                        .apply(find_real_start_balance).reset_index()
    
    # Merge the computed start_balance back into txn_agg.
    txn_agg = pd.merge(txn_agg, real_starts, on='prism_consumer_id', how='left')
    txn_agg['monthly_balance'] = txn_agg['cumulative_inflow'] + txn_agg['start_balance']
    monthly_balance_df = txn_agg[['prism_consumer_id', 'year_month', 'monthly_balance']]
    return monthly_balance_df

def compute_balance_std(monthly_balance_df):
    """
    Compute the standard deviation of monthly balance for each consumer.
    
    Args:
        monthly_balance_df (pd.DataFrame): DataFrame with monthly balances.
        
    Returns:
        pd.DataFrame: DataFrame with columns 'prism_consumer_id' and 'balance_std'.
    """
    balance_std_df = monthly_balance_df.groupby('prism_consumer_id')\
                                       .agg({'monthly_balance': 'std'})\
                                       .reset_index().rename(columns={'monthly_balance': 'balance_std'})
    return balance_std_df

def generate_income_df(txn_df, income_categories=None):
    """
    Generate an income DataFrame by selecting transactions in income categories.
    
    Args:
        txn_df (pd.DataFrame): Transaction data.
        income_categories (list, optional): List of categories to treat as income.
            Defaults to a preset list.
    
    Returns:
        pd.DataFrame: DataFrame with columns 'prism_consumer_id', 'year_month', and 'income'.
    """
    if income_categories is None:
        income_categories = ['PAYCHECK', 'DEPOSIT', 'INVESTMENT', 
                             'EXTERNAL_TRANSFER', 'UNEMPLOYMENT_BENEFITS', 'OTHER_BENEFITS']
    temp = txn_df[txn_df['category'].isin(income_categories)]
    temp_cred = temp[temp['credit_or_debit'] == 'CREDIT']
    income_df = temp_cred.groupby(['prism_consumer_id', 'year_month'])\
                         .agg({'amount': 'sum'}).reset_index().rename(columns={'amount': 'income'})
    return income_df

def compute_average_income(income_df):
    """
    Compute the average income per consumer.
    
    Args:
        income_df (pd.DataFrame): Income DataFrame.
        
    Returns:
        pd.DataFrame: DataFrame with columns 'prism_consumer_id' and 'income'.
    """
    avg_income = income_df.groupby('prism_consumer_id')['income']\
                          .mean().reset_index()
    return avg_income

# -------------------------
# VISUALIZATION FUNCTIONS
# -------------------------

def plot_balance_timeseries(balance_dq, sample_size_good=3, sample_size_bad=3):
    """
    Plot sample consumer monthly balances over time, separating good and bad consumers.
    
    Args:
        balance_dq (pd.DataFrame): DataFrame containing monthly balances merged with DQ_TARGET.
        sample_size_good (int): Number of good consumers to sample.
        sample_size_bad (int): Number of bad consumers to sample.
    """
    good_ids = balance_dq.loc[balance_dq['DQ_TARGET'] == 0, 'prism_consumer_id'].unique()
    bad_ids  = balance_dq.loc[balance_dq['DQ_TARGET'] == 1, 'prism_consumer_id'].unique()
    np.random.seed(3)
    sample_good = np.random.choice(good_ids, size=sample_size_good, replace=False)
    sample_bad  = np.random.choice(bad_ids, size=sample_size_bad, replace=False)
    sample_ids = list(sample_good) + list(sample_bad)
    
    df_plot = balance_dq[balance_dq['prism_consumer_id'].isin(sample_ids)].copy()
    df_plot['year_month'] = df_plot['year_month'].dt.to_timestamp()
    df_plot = df_plot.sort_values(['prism_consumer_id', 'year_month'])
    
    plt.figure(figsize=(10, 6))
    sns.lineplot(
        data=df_plot,
        x='year_month',
        y='monthly_balance',
        hue='prism_consumer_id',
        style='DQ_TARGET',
        markers=True,
        dashes=False
    )
    plt.title('Sample Consumers: Balances Over Time')
    plt.xlabel('Year-Month')
    plt.ylabel('Monthly Balance')
    plt.tight_layout()
    plt.show()

def plot_boxplot_feature_by_target(train_df, feature_df, feature_column):
    """
    Plot a boxplot of a specified feature split by DQ_TARGET.
    
    Args:
        train_df (pd.DataFrame): Training DataFrame with 'prism_consumer_id' and 'DQ_TARGET'.
        feature_df (pd.DataFrame): DataFrame with 'prism_consumer_id' and the feature.
        feature_column (str): Name of the feature column to plot.
    """
    merged = train_df.merge(feature_df, on='prism_consumer_id', how='left').dropna()
    merged = merged.drop(columns=['prism_consumer_id'])
    plt.figure(figsize=(10, 6))
    data_good = merged[merged['DQ_TARGET'] == 0][feature_column]
    data_bad  = merged[merged['DQ_TARGET'] == 1][feature_column]
    plt.boxplot([data_good, data_bad], labels=['DQ_TARGET = 0', 'DQ_TARGET = 1'])
    plt.title(f'{feature_column} by DQ_TARGET')
    plt.ylabel(feature_column)
    plt.grid(True)
    plt.show()

# -------------------------
# MODEL TRAINING FUNCTIONS
# -------------------------

from imblearn.over_sampling import SMOTE
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, classification_report, roc_auc_score

def apply_smote_to_data(X_train, y_train, random_state=42):
    """
    Apply SMOTE to balance the training dataset.
    
    Args:
        X_train (pd.DataFrame): Training features.
        y_train (pd.Series): Training labels.
        random_state (int): Random state for reproducibility.
        
    Returns:
        tuple: Resampled (X_train, y_train).
    """
    smote = SMOTE(random_state=random_state)
    X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)
    print("\nTraining set class distribution after SMOTE:")
    print(pd.Series(y_train_resampled).value_counts(normalize=True))
    return X_train_resampled, y_train_resampled

def train_logistic_regression_model(feature_df, train_df, target_col='DQ_TARGET', test_size=0.25, random_state=1):
    """
    Train and evaluate a logistic regression model using the provided feature DataFrame.
    
    Args:
        feature_df (pd.DataFrame): DataFrame containing 'prism_consumer_id' and feature(s).
        train_df (pd.DataFrame): Training DataFrame with 'prism_consumer_id' and target variable.
        target_col (str): Target variable column name.
        test_size (float): Proportion for the test split.
        random_state (int): Random state for reproducibility.
    """
    merged = train_df.merge(feature_df, on='prism_consumer_id', how='left').dropna()
    y = merged[target_col]
    X = merged.drop(columns=['prism_consumer_id', target_col])
    
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=test_size, random_state=random_state)
    X_train, y_train = apply_smote_to_data(X_train, y_train, random_state=random_state)
    
    clf = LogisticRegression(random_state=0, max_iter=2000)
    clf.fit(X_train, y_train)
    
    y_pred = clf.predict(X_test)
    y_proba = clf.predict_proba(X_test)[:, 1]
    accuracy = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred, average='weighted')
    auc = roc_auc_score(y_test, y_proba)
    
    print("Accuracy:", accuracy)
    print("F1 Score:", f1)
    print("AUC-ROC Score:", auc)
    print("Classification Report:\n", classification_report(y_test, y_pred))

def split_data(X, y, test_size=0.2, random_state=42):
    """
    Split the data into training and testing sets.
    
    Args:
        X (pd.DataFrame): Feature matrix.
        y (pd.Series): Target variable.
        test_size (float): Proportion of test data.
        random_state (int): Random state for reproducibility.
    
    Returns:
        tuple: (X_train, X_test, y_train, y_test).
    """
    # Ensure there are no NaN values in the features.
    X = X.fillna(0)
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, random_state=random_state, stratify=y
    )
    print("Training set class distribution before SMOTE:")
    print(pd.Series(y_train).value_counts(normalize=True))
    return X_train, X_test, y_train, y_test

def merge_all_features(feature_dfs):
    """
    Merge multiple feature DataFrames on 'prism_consumer_id' into one giant DataFrame.
    
    Args:
        feature_dfs (list of pd.DataFrame): List of feature DataFrames that each include the 'prism_consumer_id' column.
        
    Returns:
        pd.DataFrame: A merged DataFrame containing all features.
    """
    from functools import reduce
    merged_features = reduce(lambda left, right: pd.merge(left, right, on='prism_consumer_id', how='outer'),
                               feature_dfs)
    return merged_features

def prepare_features(cons_df, acct_df, txn_df):
    """
    Prepare a simple feature set from consumer, account, and transaction data.
    
    This example extracts:
      - Consumer credit score.
      - Account-level statistics (average, max, min, std of balance).
      - Transaction-level statistics (count, average, total amount, credit ratio).
    
    Returns:
        tuple: (X, y) where X is the feature matrix and y is the target variable.
    """
    valid_cons = cons_df.dropna(subset=['DQ_TARGET'])
    features = pd.DataFrame()
    features['prism_consumer_id'] = valid_cons['prism_consumer_id']
    features['credit_score'] = valid_cons['credit_score']
    
    account_features = acct_df.groupby('prism_consumer_id').agg({
        'balance': ['mean', 'max', 'min', 'std']
    }).reset_index()
    account_features.columns = ['prism_consumer_id', 'avg_balance', 'max_balance', 
                              'min_balance', 'std_balance']
    
    transaction_features = txn_df.groupby('prism_consumer_id').agg({
        'amount': ['count', 'mean', 'sum'],
        'credit_or_debit': lambda x: (x == 'CREDIT').mean()
    }).reset_index()
    transaction_features.columns = ['prism_consumer_id', 'transaction_count', 
                                  'avg_transaction', 'total_transactions', 'credit_ratio']
    
    features = features.merge(account_features, on='prism_consumer_id', how='left')
    features = features.merge(transaction_features, on='prism_consumer_id', how='left')
    
    # Fill any missing values to ensure SMOTE does not receive NaNs.
    X = features.fillna(0)
    
    return X

In [22]:
#!/usr/bin/env python
"""
Refactored script for consumer account and transaction analysis,
feature engineering, visualization, and model training.

Data files expected:
  - data/q2-ucsd-acctDF.pqt
  - data/q2-ucsd-consDF.pqt
  - data/q2-ucsd-trxnDF.pqt
  - data/q2-ucsd-cat-map.csv
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import time

# -------------------------
# DATA LOADING & PREPROCESSING
# -------------------------

def load_data():
    """
    Load account, consumer, transaction, and category mapping data.
    
    Returns:
        tuple: (acct_df, cons_df, txn_df, cat_map_df)
    """
    acct_df = pd.read_parquet("data/q2-ucsd-acctDF.pqt")
    cons_df = pd.read_parquet("data/q2-ucsd-consDF.pqt")
    txn_df  = pd.read_parquet("data/q2-ucsd-trxnDF.pqt")
    cat_map_df = pd.read_csv("data/q2-ucsd-cat-map.csv")
    return acct_df, cons_df, txn_df, cat_map_df

def create_category_map(cat_map_df):
    """
    Create a dictionary mapping from numeric category code to category string.
    
    Args:
        cat_map_df (pd.DataFrame): DataFrame with a 'category' column.
        
    Returns:
        dict: Mapping of numeric code -> category.
    """
    return {i: cat for i, cat in enumerate(cat_map_df['category'])}

def preprocess_transactions(txn_df, cat_map):
    """
    Preprocess transaction data by mapping numeric categories to strings and adding a 'year_month' column.
    
    Args:
        txn_df (pd.DataFrame): Transaction data.
        cat_map (dict): Mapping for the 'category' column.
    
    Returns:
        pd.DataFrame: Processed transaction DataFrame.
    """
    txn_df = txn_df.copy()
    txn_df['category'] = txn_df['category'].apply(lambda x: cat_map[x])
    txn_df['year_month'] = pd.to_datetime(txn_df['posted_date']).dt.to_period('M')
    return txn_df

def filter_transactions_by_consumers(txn_df, consumer_ids):
    """
    Filter transaction data to only include rows for given consumer IDs.
    
    Args:
        txn_df (pd.DataFrame): Transaction data.
        consumer_ids (set): Set of consumer IDs to keep.
        
    Returns:
        pd.DataFrame: Filtered transaction DataFrame.
    """
    return txn_df[txn_df['prism_consumer_id'].isin(consumer_ids)].copy()

def get_train_df(cons_df):
    """
    Extract the training DataFrame containing consumers with non-null DQ_TARGET.
    
    Args:
        cons_df (pd.DataFrame): Consumer-level data.
    
    Returns:
        pd.DataFrame: DataFrame with columns 'prism_consumer_id' and 'DQ_TARGET'.
    """
    return cons_df[cons_df['DQ_TARGET'].notna()][['prism_consumer_id', 'DQ_TARGET']].copy()

# -------------------------
# FEATURE ENGINEERING
# -------------------------

def aggregate_transactions(txn_df):
    """
    Aggregate transactions by consumer and month to compute monthly income,
    spending, and net income.
    
    Returns:
        pd.DataFrame: DataFrame with columns:
            'prism_consumer_id', 'year_month', 'monthly_income',
            'monthly_spending', 'monthly_net_income'.
    """
    income = txn_df[txn_df['credit_or_debit'] == 'CREDIT']\
                .groupby(['prism_consumer_id', 'year_month'])['amount']\
                .sum().rename('monthly_income')
    spending = txn_df[txn_df['credit_or_debit'] == 'DEBIT']\
                .groupby(['prism_consumer_id', 'year_month'])['amount']\
                .sum().rename('monthly_spending')
    txn_agg = pd.concat([income, spending], axis=1).fillna(0).reset_index()
    txn_agg['monthly_net_income'] = txn_agg['monthly_income'] - txn_agg['monthly_spending']
    return txn_agg

def zscoring_normalize(df, column):
    """
    Apply Z-score normalization to a specified column and add a new column with suffix '_z_score'.
    
    Args:
        df (pd.DataFrame): DataFrame containing the column.
        column (str): Column name to normalize.
        
    Returns:
        pd.DataFrame: DataFrame with an additional column named '<column>_z_score'.
    """
    df = df.copy()
    mean_val = df[column].mean()
    std_val  = df[column].std()
    df[column + '_z_score'] = (df[column] - mean_val) / std_val
    return df

def compute_average_nmi(txn_agg):
    """
    Compute the average net monthly income (NMI) for each consumer and apply Z-score normalization.
    
    Args:
        txn_agg (pd.DataFrame): Aggregated transaction data.
        
    Returns:
        pd.DataFrame: DataFrame with 'prism_consumer_id', 'average_nmi', and 'average_nmi_z_score'.
    """
    avg_nmi = txn_agg.groupby('prism_consumer_id')['monthly_net_income']\
                     .mean().reset_index().rename(columns={'monthly_net_income': 'average_nmi'})
    avg_nmi = zscoring_normalize(avg_nmi, 'average_nmi')
    return avg_nmi

def compute_balance_features(acct_df, txn_agg):
    """
    Compute monthly balance features using account data and aggregated transaction data.
    
    Steps:
      1. Convert account balance dates to a monthly period and aggregate balances.
      2. Compute cumulative monthly net inflow for each consumer.
      3. Merge the aggregated transactions with account balance on consumer ID and month.
      4. For each consumer, determine a 'start_balance' using the earliest available balance (anchor)
         and the inflows prior to that anchor.
      5. Compute monthly balance as: monthly_balance = cumulative_inflow + start_balance.
    
    Returns:
        pd.DataFrame: DataFrame with columns 'prism_consumer_id', 'year_month', and 'monthly_balance'.
    """
    acct_df = acct_df.copy()
    acct_df['year_month'] = pd.to_datetime(acct_df['balance_date']).dt.to_period('M')
    balance_df = acct_df.groupby(['prism_consumer_id', 'year_month'])\
                        .agg({'balance': 'sum'}).reset_index()
    
    txn_agg = txn_agg.sort_values('year_month')
    txn_agg['cumulative_inflow'] = txn_agg.groupby('prism_consumer_id')['monthly_net_income'].cumsum()
    
    merged = pd.merge(txn_agg, balance_df, on=['prism_consumer_id', 'year_month'], how='left')
    
    def find_real_start_balance(grp):
        """
        For a given consumer group, find the earliest available balance (anchor)
        and compute the consumer's starting balance.
        """
        anchor_rows = grp.dropna(subset=["balance"]).sort_values("year_month")
        if anchor_rows.empty:
            return pd.Series({"start_balance": 0.0})
        anchor_balance = anchor_rows["balance"].iloc[0]
        anchor_month   = anchor_rows["year_month"].iloc[0]
        inflow_before_anchor = grp.loc[grp["year_month"] <= anchor_month, "monthly_net_income"].sum()
        start_balance = anchor_balance - inflow_before_anchor
        return pd.Series({"start_balance": start_balance})
    
    real_starts = merged.groupby("prism_consumer_id", group_keys=False)\
                        .apply(find_real_start_balance).reset_index()
    
    txn_agg = pd.merge(txn_agg, real_starts, on='prism_consumer_id', how='left')
    txn_agg['monthly_balance'] = txn_agg['cumulative_inflow'] + txn_agg['start_balance']
    monthly_balance_df = txn_agg[['prism_consumer_id', 'year_month', 'monthly_balance']]
    return monthly_balance_df

def compute_balance_std(monthly_balance_df):
    """
    Compute the standard deviation of monthly balance for each consumer.
    
    Args:
        monthly_balance_df (pd.DataFrame): DataFrame with monthly balances.
        
    Returns:
        pd.DataFrame: DataFrame with columns 'prism_consumer_id' and 'balance_std'.
    """
    balance_std_df = monthly_balance_df.groupby('prism_consumer_id')\
                                       .agg({'monthly_balance': 'std'})\
                                       .reset_index().rename(columns={'monthly_balance': 'balance_std'})
    return balance_std_df

def generate_income_df(txn_df, income_categories=None):
    """
    Generate an income DataFrame by selecting transactions in income categories.
    
    Args:
        txn_df (pd.DataFrame): Transaction data.
        income_categories (list, optional): List of categories to treat as income.
            Defaults to a preset list.
    
    Returns:
        pd.DataFrame: DataFrame with columns 'prism_consumer_id', 'year_month', and 'income'.
    """
    if income_categories is None:
        income_categories = ['PAYCHECK', 'DEPOSIT', 'INVESTMENT', 
                             'EXTERNAL_TRANSFER', 'UNEMPLOYMENT_BENEFITS', 'OTHER_BENEFITS']
    temp = txn_df[txn_df['category'].isin(income_categories)]
    temp_cred = temp[temp['credit_or_debit'] == 'CREDIT']
    income_df = temp_cred.groupby(['prism_consumer_id', 'year_month'])\
                         .agg({'amount': 'sum'}).reset_index().rename(columns={'amount': 'income'})
    return income_df

def compute_average_income(income_df):
    """
    Compute the average income per consumer.
    
    Args:
        income_df (pd.DataFrame): Income DataFrame.
        
    Returns:
        pd.DataFrame: DataFrame with columns 'prism_consumer_id' and 'income'.
    """
    avg_income = income_df.groupby('prism_consumer_id')['income']\
                          .mean().reset_index()
    return avg_income

# -------------------------
# FEATURE MERGING FUNCTIONS
# -------------------------

def merge_all_features(feature_dfs):
    """
    Merge multiple feature DataFrames on 'prism_consumer_id' into one giant DataFrame.
    
    Args:
        feature_dfs (list of pd.DataFrame): List of feature DataFrames that each include the 'prism_consumer_id' column.
        
    Returns:
        pd.DataFrame: A merged DataFrame containing all features.
    """
    from functools import reduce
    merged_features = reduce(lambda left, right: pd.merge(left, right, on='prism_consumer_id', how='outer'),
                               feature_dfs)
    return merged_features

def prepare_features_df(cons_df, acct_df, txn_df):
    """
    Prepare a feature DataFrame from consumer, account, and transaction data.
    Retains 'prism_consumer_id' for merging with other feature sets.
    
    Returns:
        pd.DataFrame: DataFrame with 'prism_consumer_id' and various engineered features.
    """
    valid_cons = cons_df.dropna(subset=['DQ_TARGET'])
    features = pd.DataFrame()
    features['prism_consumer_id'] = valid_cons['prism_consumer_id']
    features['credit_score'] = valid_cons['credit_score']
    
    account_features = acct_df.groupby('prism_consumer_id').agg({
        'balance': ['mean', 'max', 'min', 'std']
    }).reset_index()
    account_features.columns = ['prism_consumer_id', 'avg_balance', 'max_balance', 'min_balance', 'std_balance']
    
    transaction_features = txn_df.groupby('prism_consumer_id').agg({
        'amount': ['count', 'mean', 'sum'],
        'credit_or_debit': lambda x: (x == 'CREDIT').mean()
    }).reset_index()
    transaction_features.columns = ['prism_consumer_id', 'transaction_count', 
                                    'avg_transaction', 'total_transactions', 'credit_ratio']
    
    features = features.merge(account_features, on='prism_consumer_id', how='left')
    features = features.merge(transaction_features, on='prism_consumer_id', how='left')
    features = features.fillna(0)
    return features

# -------------------------
# VISUALIZATION FUNCTIONS
# -------------------------

def plot_balance_timeseries(balance_dq, sample_size_good=3, sample_size_bad=3):
    """
    Plot sample consumer monthly balances over time, separating good and bad consumers.
    
    Args:
        balance_dq (pd.DataFrame): DataFrame containing monthly balances merged with DQ_TARGET.
        sample_size_good (int): Number of good consumers to sample.
        sample_size_bad (int): Number of bad consumers to sample.
    """
    good_ids = balance_dq.loc[balance_dq['DQ_TARGET'] == 0, 'prism_consumer_id'].unique()
    bad_ids  = balance_dq.loc[balance_dq['DQ_TARGET'] == 1, 'prism_consumer_id'].unique()
    np.random.seed(3)
    sample_good = np.random.choice(good_ids, size=sample_size_good, replace=False)
    sample_bad  = np.random.choice(bad_ids, size=sample_size_bad, replace=False)
    sample_ids = list(sample_good) + list(sample_bad)
    
    df_plot = balance_dq[balance_dq['prism_consumer_id'].isin(sample_ids)].copy()
    df_plot['year_month'] = df_plot['year_month'].dt.to_timestamp()
    df_plot = df_plot.sort_values(['prism_consumer_id', 'year_month'])
    
    plt.figure(figsize=(10, 6))
    sns.lineplot(
        data=df_plot,
        x='year_month',
        y='monthly_balance',
        hue='prism_consumer_id',
        style='DQ_TARGET',
        markers=True,
        dashes=False
    )
    plt.title('Sample Consumers: Balances Over Time')
    plt.xlabel('Year-Month')
    plt.ylabel('Monthly Balance')
    plt.tight_layout()
    plt.show()

def plot_boxplot_feature_by_target(train_df, feature_df, feature_column):
    """
    Plot a boxplot of a specified feature split by DQ_TARGET.
    
    Args:
        train_df (pd.DataFrame): Training DataFrame with 'prism_consumer_id' and 'DQ_TARGET'.
        feature_df (pd.DataFrame): DataFrame with 'prism_consumer_id' and the feature.
        feature_column (str): Name of the feature column to plot.
    """
    merged = train_df.merge(feature_df, on='prism_consumer_id', how='left').dropna()
    merged = merged.drop(columns=['prism_consumer_id'])
    plt.figure(figsize=(10, 6))
    data_good = merged[merged['DQ_TARGET'] == 0][feature_column]
    data_bad  = merged[merged['DQ_TARGET'] == 1][feature_column]
    plt.boxplot([data_good, data_bad], labels=['DQ_TARGET = 0', 'DQ_TARGET = 1'])
    plt.title(f'{feature_column} by DQ_TARGET')
    plt.ylabel(feature_column)
    plt.grid(True)
    plt.show()

# -------------------------
# MODEL TRAINING FUNCTIONS
# -------------------------

from imblearn.over_sampling import SMOTE
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, classification_report, roc_auc_score

def apply_smote_to_data(X_train, y_train, random_state=42):
    """
    Apply SMOTE to balance the training dataset.
    
    Args:
        X_train (pd.DataFrame): Training features.
        y_train (pd.Series): Training labels.
        random_state (int): Random state for reproducibility.
        
    Returns:
        tuple: Resampled (X_train, y_train).
    """
    smote = SMOTE(random_state=random_state)
    X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)
    print("\nTraining set class distribution after SMOTE:")
    print(pd.Series(y_train_resampled).value_counts(normalize=True))
    return X_train_resampled, y_train_resampled

def train_logistic_regression_model(feature_df, train_df, target_col='DQ_TARGET', test_size=0.25, random_state=1):
    """
    Train and evaluate a logistic regression model using the provided feature DataFrame.
    
    Args:
        feature_df (pd.DataFrame): DataFrame containing 'prism_consumer_id' and feature(s).
        train_df (pd.DataFrame): Training DataFrame with 'prism_consumer_id' and target variable.
        target_col (str): Target variable column name.
        test_size (float): Proportion for the test split.
        random_state (int): Random state for reproducibility.
    """
    merged = train_df.merge(feature_df, on='prism_consumer_id', how='left').dropna()
    y = merged[target_col]
    X = merged.drop(columns=['prism_consumer_id', target_col])
    
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=test_size, random_state=random_state)
    X_train, y_train = apply_smote_to_data(X_train, y_train, random_state=random_state)
    
    clf = LogisticRegression(random_state=0, max_iter=2000)
    clf.fit(X_train, y_train)
    
    y_pred = clf.predict(X_test)
    y_proba = clf.predict_proba(X_test)[:, 1]
    accuracy = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred, average='weighted')
    auc = roc_auc_score(y_test, y_proba)
    
    print("Logistic Regression Model:")
    print("Accuracy:", accuracy)
    print("F1 Score:", f1)
    print("AUC-ROC Score:", auc)
    print("Classification Report:\n", classification_report(y_test, y_pred))

# --- New Model Functions ---

def train_random_forest_model(feature_df, train_df, target_col='DQ_TARGET', test_size=0.25, random_state=1):
    """
    Train and evaluate a Random Forest model.
    Measures training and scoring time, and prints performance metrics.
    """
    from sklearn.ensemble import RandomForestClassifier
    merged = train_df.merge(feature_df, on='prism_consumer_id', how='left').dropna()
    y = merged[target_col]
    X = merged.drop(columns=['prism_consumer_id', target_col])
    
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=test_size, random_state=random_state)
    X_train, y_train = apply_smote_to_data(X_train, y_train, random_state=random_state)
    
    rf_clf = RandomForestClassifier(random_state=random_state, n_estimators=100)
    
    start_train = time.time()
    rf_clf.fit(X_train, y_train)
    training_time = time.time() - start_train
    
    start_score = time.time()
    y_pred = rf_clf.predict(X_test)
    y_proba = rf_clf.predict_proba(X_test)[:, 1]
    scoring_time = time.time() - start_score
    
    accuracy = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred, average='weighted')
    auc = roc_auc_score(y_test, y_proba)
    
    print("Random Forest Model:")
    print(f"Training time: {training_time:.4f} seconds")
    print(f"Scoring time: {scoring_time:.4f} seconds")
    print("Accuracy:", accuracy)
    print("F1 Score:", f1)
    print("AUC-ROC Score:", auc)
    print("Classification Report:\n", classification_report(y_test, y_pred))

def train_gradient_boosting_model(feature_df, train_df, target_col='DQ_TARGET', test_size=0.25, random_state=1):
    """
    Train and evaluate a Gradient Boosting model.
    Measures training and scoring time, and prints performance metrics.
    """
    from sklearn.ensemble import GradientBoostingClassifier
    merged = train_df.merge(feature_df, on='prism_consumer_id', how='left').dropna()
    y = merged[target_col]
    X = merged.drop(columns=['prism_consumer_id', target_col])
    
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=test_size, random_state=random_state)
    X_train, y_train = apply_smote_to_data(X_train, y_train, random_state=random_state)
    
    gb_clf = GradientBoostingClassifier(random_state=random_state)
    
    start_train = time.time()
    gb_clf.fit(X_train, y_train)
    training_time = time.time() - start_train
    
    start_score = time.time()
    y_pred = gb_clf.predict(X_test)
    y_proba = gb_clf.predict_proba(X_test)[:, 1]
    scoring_time = time.time() - start_score
    
    accuracy = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred, average='weighted')
    auc = roc_auc_score(y_test, y_proba)
    
    print("Gradient Boosting Model:")
    print(f"Training time: {training_time:.4f} seconds")
    print(f"Scoring time: {scoring_time:.4f} seconds")
    print("Accuracy:", accuracy)
    print("F1 Score:", f1)
    print("AUC-ROC Score:", auc)
    print("Classification Report:\n", classification_report(y_test, y_pred))

def train_xgboost_model(feature_df, train_df, target_col='DQ_TARGET', test_size=0.25, random_state=1):
    """
    Train and evaluate an XGBoost model.
    Measures training and scoring time, and prints performance metrics.
    """
    import xgboost as xgb
    merged = train_df.merge(feature_df, on='prism_consumer_id', how='left').dropna()
    y = merged[target_col]
    X = merged.drop(columns=['prism_consumer_id', target_col])
    
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=test_size, random_state=random_state)
    X_train, y_train = apply_smote_to_data(X_train, y_train, random_state=random_state)
    
    xgb_clf = xgb.XGBClassifier(random_state=random_state, use_label_encoder=False, eval_metric='logloss')
    
    start_train = time.time()
    xgb_clf.fit(X_train, y_train)
    training_time = time.time() - start_train
    
    start_score = time.time()
    y_pred = xgb_clf.predict(X_test)
    y_proba = xgb_clf.predict_proba(X_test)[:, 1]
    scoring_time = time.time() - start_score
    
    accuracy = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred, average='weighted')
    auc = roc_auc_score(y_test, y_proba)
    
    print("XGBoost Model:")
    print(f"Training time: {training_time:.4f} seconds")
    print(f"Scoring time: {scoring_time:.4f} seconds")
    print("Accuracy:", accuracy)
    print("F1 Score:", f1)
    print("AUC-ROC Score:", auc)
    print("Classification Report:\n", classification_report(y_test, y_pred))

# -------------------------
# ADDITIONAL FEATURE PREPARATION FOR MODELS
# -------------------------

def prepare_features(cons_df, acct_df, txn_df):
    """
    Prepare a feature set from consumer, account, and transaction data for modeling.
    This version drops the consumer ID.
    
    Returns:
        tuple: (X, y) where X is the feature matrix (without 'prism_consumer_id') and y is the target variable.
    """
    features = prepare_features_df(cons_df, acct_df, txn_df)
    y = cons_df.dropna(subset=['DQ_TARGET'])['DQ_TARGET']
    X = features.drop(columns=['prism_consumer_id'])
    return X, y

In [23]:
# -------------------------
# MAIN EXECUTION NEW VERSION
# -------------------------

# -------------------------
# Load Data
# -------------------------
acct_df, cons_df, txn_df, cat_map_df = load_data()

# -------------------------
# Preprocess Transactions
# -------------------------
cat_map = create_category_map(cat_map_df)
txn_df = preprocess_transactions(txn_df, cat_map)

# Keep only transactions for consumers present in the training set.
train_df = get_train_df(cons_df)
txn_df = filter_transactions_by_consumers(txn_df, set(train_df['prism_consumer_id']))

# -------------------------
# Feature Engineering
# -------------------------
txn_agg = aggregate_transactions(txn_df)
avg_nmi = compute_average_nmi(txn_agg)
avg_nmi_feat = avg_nmi[['prism_consumer_id', 'average_nmi_z_score']]

monthly_balance_df = compute_balance_features(acct_df, txn_agg)
balance_std_df = compute_balance_std(monthly_balance_df)

income_df = generate_income_df(txn_df)
avg_income = compute_average_income(income_df)

basic_features = prepare_features_df(cons_df, acct_df, txn_df)

# -------------------------
# Merge All Feature DataFrames
# -------------------------
all_features = merge_all_features([avg_nmi, balance_std_df, avg_income, basic_features])
print("Merged features shape:", all_features.shape)

# -------------------------
# Model Training with Different Feature Sets
# -------------------------
print("Logistic Regression using Normalized Average Net Monthly Inflow:")
train_logistic_regression_model(avg_nmi_feat, train_df)

print("\nLogistic Regression using Normalized Account Balance Sum:")
acct_balance_df = acct_df.groupby("prism_consumer_id")\
                            .agg(balance_sum=('balance', 'sum')).reset_index()
acct_balance_df = zscoring_normalize(acct_balance_df, 'balance_sum')
acct_balance_feat = acct_balance_df[['prism_consumer_id', 'balance_sum_z_score']]
train_logistic_regression_model(acct_balance_feat, train_df)

print("\nLogistic Regression using Balance Standard Deviation:")
train_logistic_regression_model(balance_std_df, train_df)

print("\nLogistic Regression using Average Income:")
train_logistic_regression_model(avg_income, train_df)

print("\nLogistic Regression using Merged Features:")
train_logistic_regression_model(all_features, train_df)

# -------------------------
# Compare Three Alternative Models on Merged Features
# -------------------------
print("\n--- Model Comparison on Merged Features ---")

print("\nRandom Forest Model:")
train_random_forest_model(all_features, train_df)

print("\nGradient Boosting Model:")
train_gradient_boosting_model(all_features, train_df)

print("\nXGBoost Model:")
train_xgboost_model(all_features, train_df)


/var/folders/2f/pp48ygln7m9cxl6lrzd_n_t40000gn/T/ipykernel_99994/173411596.py:185: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(find_real_start_balance).reset_index()


Merged features shape: (12000, 14)
Logistic Regression using Normalized Average Net Monthly Inflow:

Training set class distribution after SMOTE:
DQ_TARGET
0.0    0.5
1.0    0.5
Name: proportion, dtype: float64
Logistic Regression Model:
Accuracy: 0.4019303688383316
F1 Score: 0.4985180719954342
AUC-ROC Score: 0.5629111292146444
Classification Report:
               precision    recall  f1-score   support

         0.0       0.95      0.37      0.53      2653
         1.0       0.10      0.79      0.18       248

    accuracy                           0.40      2901
   macro avg       0.53      0.58      0.36      2901
weighted avg       0.88      0.40      0.50      2901


Logistic Regression using Normalized Account Balance Sum:

Training set class distribution after SMOTE:
DQ_TARGET
0.0    0.5
1.0    0.5
Name: proportion, dtype: float64
Logistic Regression Model:
Accuracy: 0.4089162182936203
F1 Score: 0.5025208436123586
AUC-ROC Score: 0.7171672977142056
Classification Report:
       

/Users/qianjin/anaconda3/envs/creditrisk/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:469: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Logistic Regression Model:
Accuracy: 0.7144948755490483
F1 Score: 0.7748689039843721
AUC-ROC Score: 0.7533222591362125
Classification Report:
               precision    recall  f1-score   support

         0.0       0.96      0.72      0.82      2494
         1.0       0.18      0.65      0.28       238

    accuracy                           0.71      2732
   macro avg       0.57      0.69      0.55      2732
weighted avg       0.89      0.71      0.77      2732


--- Model Comparison on Merged Features ---

Random Forest Model:

Training set class distribution after SMOTE:
DQ_TARGET
0.0    0.5
1.0    0.5
Name: proportion, dtype: float64
Random Forest Model:
Training time: 2.8418 seconds
Scoring time: 0.0501 seconds
Accuracy: 0.8323572474377745
F1 Score: 0.8522189438283977
AUC-ROC Score: 0.7530265915508143
Classification Report:
               precision    recall  f1-score   support

         0.0       0.94      0.87      0.90      2494
         1.0       0.24      0.42      0.30    

/Users/qianjin/anaconda3/envs/creditrisk/lib/python3.12/site-packages/xgboost/core.py:158: UserWarning: [23:29:14] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)


In [ ]:
# -------------------------
# MAIN EXECUTION
# -------------------------

# -------------------------
# Load Data
# -------------------------
acct_df, cons_df, txn_df, cat_map_df = load_data()

# -------------------------
# Preprocess Transactions
# -------------------------
cat_map = create_category_map(cat_map_df)
txn_df = preprocess_transactions(txn_df, cat_map)

# Only keep transactions for consumers present in the training set.
train_df = get_train_df(cons_df)
txn_df = filter_transactions_by_consumers(txn_df, set(train_df['prism_consumer_id']))

# -------------------------
# Feature Engineering
# -------------------------
# 1. Transaction Aggregation and Average Net Monthly Income
txn_agg = aggregate_transactions(txn_df)
avg_nmi = compute_average_nmi(txn_agg)
# Use only the normalized feature for modeling.
avg_nmi_feat = avg_nmi[['prism_consumer_id', 'average_nmi_z_score']]

# 2. Balance Features: Compute monthly balances and then per-consumer balance standard deviation.
monthly_balance_df = compute_balance_features(acct_df, txn_agg)
balance_std_df = compute_balance_std(monthly_balance_df)

# 3. Income Features
income_df = generate_income_df(txn_df)
avg_income = compute_average_income(income_df)

# 4. All features



# -------------------------
# (Optional) Visualization
# -------------------------
# Merge monthly balances with training target for visualization.
balance_dq = monthly_balance_df.merge(train_df, on='prism_consumer_id', how='left')
# Uncomment the line below to view balance trends.
# plot_balance_timeseries(balance_dq)

# Uncomment to plot a boxplot of account balance sum by DQ_TARGET.
# acct_balance_df = acct_df.groupby("prism_consumer_id").agg(balance_sum=('balance', 'sum')).reset_index()
# acct_balance_df = zscoring_normalize(acct_balance_df, 'balance_sum')
# plot_boxplot_feature_by_target(train_df, acct_balance_df[['prism_consumer_id', 'balance_sum_z_score']], 'balance_sum_z_score')

# -------------------------
# Model Training
# -------------------------
print("Logistic Regression using Normalized Average Net Monthly Inflow:")
train_logistic_regression_model(avg_nmi_feat, train_df)

# Account Balance Sum as a feature.
acct_balance_df = acct_df.groupby("prism_consumer_id").agg(balance_sum=('balance', 'sum')).reset_index()
acct_balance_df = zscoring_normalize(acct_balance_df, 'balance_sum')
acct_balance_feat = acct_balance_df[['prism_consumer_id', 'balance_sum_z_score']]
print("\nLogistic Regression using Normalized Account Balance Sum:")
train_logistic_regression_model(acct_balance_feat, train_df)

print("\nLogistic Regression using Balance Standard Deviation:")
train_logistic_regression_model(balance_std_df, train_df)

print("\nLogistic Regression using Average Income:")
train_logistic_regression_model(avg_income, train_df)

/var/folders/2f/pp48ygln7m9cxl6lrzd_n_t40000gn/T/ipykernel_99994/683633335.py:190: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(find_real_start_balance).reset_index()


Logistic Regression using Normalized Average Net Monthly Inflow:

Training set class distribution after SMOTE:
DQ_TARGET
0.0    0.5
1.0    0.5
Name: proportion, dtype: float64
Accuracy: 0.4019303688383316
F1 Score: 0.4985180719954342
AUC-ROC Score: 0.5629111292146444
Classification Report:
               precision    recall  f1-score   support

         0.0       0.95      0.37      0.53      2653
         1.0       0.10      0.79      0.18       248

    accuracy                           0.40      2901
   macro avg       0.53      0.58      0.36      2901
weighted avg       0.88      0.40      0.50      2901


Logistic Regression using Normalized Account Balance Sum:

Training set class distribution after SMOTE:
DQ_TARGET
0.0    0.5
1.0    0.5
Name: proportion, dtype: float64
Accuracy: 0.4089162182936203
F1 Score: 0.5025208436123586
AUC-ROC Score: 0.7171672977142056
Classification Report:
               precision    recall  f1-score   support

         0.0       0.96      0.37      0

In [12]:
# -------------------------
# Additional Example: Prepare a feature set and split data
# -------------------------
X = prepare_features(cons_df, acct_df, txn_df)
X


,prism_consumer_id,credit_score,avg_balance,max_balance,min_balance,std_balance,transaction_count,avg_transaction,total_transactions,credit_ratio
0,0,726.0,160.185000,294.67,25.70,190.190511,408.0,71.802034,29295.23,0.093137
1,1,626.0,1651.210000,3211.18,91.24,2206.130731,314.0,152.873153,48002.17,0.226115
2,2,680.0,1402.680000,2561.43,243.93,1638.719965,448.0,100.668058,45099.29,0.180804
3,3,734.0,3833.505000,6690.19,976.82,4039.962670,271.0,156.779557,42487.26,0.188192
4,4,676.0,197.275000,391.62,2.93,274.845335,306.0,106.130131,32475.82,0.130719
...,...,...,...,...,...,...,...,...,...,...
11995,13995,802.0,342.933333,1028.23,0.00,593.484391,62.0,41.040484,2544.51,0.709677
11996,13996,652.0,1642.252857,8739.24,-3.57,3282.272432,696.0,154.659109,107642.74,0.244253
11997,13997,765.0,1198.425000,1396.78,1000.07,280.516331,42.0,354.723333,14898.38,0.785714
11998,13998,685.0,2967.142000,13945.68,5.00,6139.581001,301.0,334.970233,100826.04,0.289037


In [14]:
train_logistic_regression_model(X, train_df)


Training set class distribution after SMOTE:
DQ_TARGET
0.0    0.5
1.0    0.5
Name: proportion, dtype: float64
Accuracy: 0.708
F1 Score: 0.7748169363703563
AUC-ROC Score: 0.7497064667630057
Classification Report:
               precision    recall  f1-score   support

         0.0       0.96      0.71      0.82      2768
         1.0       0.16      0.64      0.25       232

    accuracy                           0.71      3000
   macro avg       0.56      0.68      0.54      3000
weighted avg       0.90      0.71      0.77      3000



In [19]:
# Merge the feature sets into one giant DataFrame (all DataFrames must have 'prism_consumer_id').
all_features = merge_all_features([avg_nmi, balance_std_df, avg_income, X])
print("Merged features shape:", all_features.shape)

Merged features shape: (12000, 14)


In [20]:
print("\nLogistic Regression using Merged Features:")
train_logistic_regression_model(all_features, train_df)


Logistic Regression using Merged Features:

Training set class distribution after SMOTE:
DQ_TARGET
0.0    0.5
1.0    0.5
Name: proportion, dtype: float64
Accuracy: 0.7144948755490483
F1 Score: 0.7748689039843721
AUC-ROC Score: 0.7533222591362125
Classification Report:
               precision    recall  f1-score   support

         0.0       0.96      0.72      0.82      2494
         1.0       0.18      0.65      0.28       238

    accuracy                           0.71      2732
   macro avg       0.57      0.69      0.55      2732
weighted avg       0.89      0.71      0.77      2732



/Users/qianjin/anaconda3/envs/creditrisk/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:469: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
